<div align="center">

# Keep compute close to your text

See when a local HTTP request costs more than encoding the text.

</div>

## When do you need this?

You are deciding whether your app should call fse locally or through a service.
You want to see which costs come from encoding and which come from transferring vectors.
**Start in the process that already has the text**, because that avoids moving text and vectors between processes.

## The story in pictures

<picture>
<source media="(prefers-color-scheme: dark)" srcset="media/07-local-compute-and-http-dark.svg">
<img src="media/07-local-compute-and-http-light.svg" alt="One path encodes text directly inside your app. Another makes an HTTP request and transfers JSON vector bytes back.">
</picture>

## 1. Understand the recorded setup

This recipe retains its **historical measurements from 1 November 2026**, rather than running a new benchmark for this rewrite.
The machine was an Apple M1 Pro with macOS 27.0.1, Python 3.13.16, and one worker.
The one-minute load average was 4.7114 before and after timing, as recorded below and in [machine provenance](run-environment.json).
A worker is a processor thread used by fse.

## 2. Load GloVe and prepare real sentences

The full GloVe table gives a 300-number vector for each STS Benchmark sentence.
The batch repeats 1,000 sentences ten times, so it does not represent a varied production workload.
The helper code below keeps temporary files and model downloads out of your project.
Follow the [index setup](README.md#run-a-recipe) before opening this recipe.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

2026-11-01 macOS-27.0.1-arm64-arm-64bit-Mach-O arm64


In [2]:
import csv
import statistics
import time
import json
import socket
import threading
import http.client
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
revision = "4e5ad402664236b85be45a0d107b95259ba8a909"
files = ["glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy"]
paths = []
for filename in files:
    cached = Path.home() / ".cache/huggingface/hub/models--fse--glove-wiki-gigaword-300/snapshots" / revision / filename
    paths.append(cached if cached.is_file() else Path(hf_hub_download("fse/glove-wiki-gigaword-300", filename, revision=revision, cache_dir=work / "hub")))
if paths[0].parent != paths[1].parent:
    paths = [Path(hf_hub_download("fse/glove-wiki-gigaword-300", filename, revision=revision, cache_dir=work / "hub")) for filename in files]
wv = KeyedVectors.load(str(paths[0]), mmap="r")
# Use the complete table, so fse_words@1 retains every valid GloVe token.
from fse.gensim import from_gensim
repository = Path.cwd().parent if Path.cwd().name == "recipes" else Path.cwd()
with (repository / "evaluation/sts-test.csv").open() as stream:
    sample = [r[5] for r in csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE)][:1000]
encoder = fse.Encoder.from_word_vectors(from_gensim(wv))
workload = sample * 10
encoder.encode(workload, workers=1)  # warm tokenizer and native pipeline
vector = encoder.encode([sample[0]], workers=1)[0]
assert vector.shape == (300,)
binary = vector.astype("<f4").tobytes()
json_bytes = json.dumps(vector.tolist(), separators=(",", ":"), allow_nan=False).encode("utf-8")
print({"binary_bytes": len(binary), "JSON_bytes": len(json_bytes), "dimensions": len(vector)})

/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr153-recipes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'binary_bytes': 1200, 'JSON_bytes': 6320, 'dimensions': 300}


## 3. Separate computation from transport

The server returns one **fixed JSON vector**, so HTTP time excludes encoding and JSON parsing.
HTTP is the request-and-response protocol used here, with a connection kept open between requests.
Only rerun this measurement on a quiet machine with permission from its other users.
The original code refuses a one-minute load average of five or more.

In [3]:
class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def setup(self):
        super().setup()
        self.connection.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(json_bytes)))
        self.end_headers()
        self.wfile.write(json_bytes)
    def log_message(self, *args):
        pass

load_before = os.getloadavg()[0]
if load_before >= 5:
    raise RuntimeError(f"load {load_before:.2f} is too high for timing")
server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
connection = http.client.HTTPConnection("127.0.0.1", server.server_port, timeout=10)
try:
    connection.request("GET", "/")
    assert connection.getresponse().read() == json_bytes
    compute, single, requests = [], [], []
    for _ in range(5):
        start = time.perf_counter_ns()
        encoder.encode(workload, workers=1)
        compute.append((time.perf_counter_ns()-start)/len(workload)/1000)
    for _ in range(100):
        start = time.perf_counter_ns()
        encoder.encode([sample[0]], workers=1)
        single.append((time.perf_counter_ns()-start)/1000)
        start = time.perf_counter_ns()
        connection.request("GET", "/")
        assert connection.getresponse().read() == json_bytes
        requests.append((time.perf_counter_ns()-start)/1000)
finally:
    connection.close()
    server.shutdown()
    server.server_close()
    thread.join()
load_after = os.getloadavg()[0]
if load_after >= 5:
    raise RuntimeError(f"load {load_after:.2f} invalidates the timing")
measurements = {
    "date": date.today().isoformat(), "machine": platform.platform(), "processor": platform.machine(),
    "table": "GloVe-300 full vocabulary", "vocabulary_rows": len(encoder.word_vectors), "workers": 1,
    "sentences_per_batch": len(workload), "batch_repetitions": 5, "single_and_HTTP_repetitions": 100,
    "compute_batch_us_per_sentence_median": statistics.median(compute),
    "compute_single_sentence_us_median": statistics.median(single),
    "HTTP_keepalive_JSON_us_median": statistics.median(requests),
    "load_before": load_before, "load_after": load_after,
    "binary_bytes": len(binary), "JSON_bytes": len(json_bytes),
}
print(json.dumps(measurements, indent=2))

{
  "date": "2026-11-01",
  "machine": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
  "processor": "arm64",
  "table": "GloVe-300 full vocabulary",
  "vocabulary_rows": 400000,
  "workers": 1,
  "sentences_per_batch": 10000,
  "batch_repetitions": 5,
  "single_and_HTTP_repetitions": 100,
  "compute_batch_us_per_sentence_median": 0.7357209,
  "compute_single_sentence_us_median": 52.313,
  "HTTP_keepalive_JSON_us_median": 178.833,
  "load_before": 4.71142578125,
  "load_after": 4.71142578125,
  "binary_bytes": 1200,
  "JSON_bytes": 6320
}


## Result

| Recorded operation | Median or size |
|---|---:|
| Encode one sentence within a 10,000-sentence batch | 0.736 µs |
| Encode one sentence in its own call | 52.3 µs |
| Read a fixed JSON response over local HTTP | 178.8 µs |
| One 300-number vector as float32 | 1,200 bytes |
| The same vector as compact JSON | 6,320 bytes |

**The local request alone cost more than one encode call in this setup.**
The batch figure shares Python call overhead across many repeated sentences.
Other networks, concurrency, and batched requests change these costs, so this does not establish a general speed ratio.

## What the sizes mean

**Float32** stores each number in four bytes, so 300 values occupy 1,200 bytes.
JSON writes numbers as text, and its size depends on their values and formatting.
The server closes at the end of the measurement cell, leaving no service running.

## What next?

- [Start with the fse 101 recipe](00-fse-101.ipynb).
- [fse API and design](../docs/architecture/runtime.md)
- [Python HTTP client](https://docs.python.org/3/library/http.client.html)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

In [4]:
scratch.cleanup()